In [1]:
!pip install -q "transformers>=5.0.0"

In [5]:
import torch
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText

model_path = "PaddlePaddle/PaddleOCR-VL-1.6"
model = AutoModelForImageTextToText.from_pretrained(
    model_path, torch_dtype=torch.float16
).to("cuda").eval()
processor = AutoProcessor.from_pretrained(model_path)
print("model load ho gaya")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


Loading weights:   0%|          | 0/608 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


model load ho gaya


In [11]:
def ocr_piece(img):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": "OCR:"},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=2048, use_cache=True)
    return processor.decode(out[0][inputs["input_ids"].shape[-1]:-1])

In [9]:
print("characters:", len(text))
print("tokens:", len(processor.tokenizer(text).input_ids))
print("----- last part -----")
print(text[-400:])

characters: 1488
tokens: 425
----- last part -----
pay the same to the lesser regularly. This lease is effective from 1st April 2008.

2. The Lessee hereby agrees to pay the above rent before 5th of every month.

3. The lessee should use the said premises for residential purpose only.

4. The rent agreement will be for a period of 12 months i.e. up to end of March 2009. But it can be extended by mutual consent by enhancing the rent at 5% increase.


In [12]:
import time

test_img = Image.open(images[0]).convert("RGB")
w, h = test_img.size
piece = test_img.crop((0, 0, w, int(w * 1.3)))

start = time.time()
text = ocr_piece(piece)
print("time:", round(time.time() - start), "sec")
print(text[:500])

time: 12 sec
This rental agreement is made and executed on this 1st day of April 2008 (1-04-08) by and between

Sri Hanumaiah
No 12, 1st Floor, 6th Cross, Balajinagar DRC Post, Bangalore 560029

Hereinafter referred to as the owner: Lesser of the one part and in favour of:

Sri Vishal Bhardwaj S/O Charnel Singh Village Pandol Road PO and Tehsil Baijnath Dist: Kangra (H.P.)
Himachal Pradesh 176125

Hereinafter referred to as the Tenant: Lessee of the other part:

Where as the terms both the lesser and the Les


In [13]:
out_dir = Path("/kaggle/working/ocr")
out_dir.mkdir(exist_ok=True)

images = list(Path("/kaggle/input").rglob("*.png"))
print("Images obtained:", len(images))

for path in images:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    page_h = int(w * 1.3)  # ek A4 page jitni height

    texts = []
    for top in range(0, h, page_h):
        piece = img.crop((0, top, w, min(top + page_h, h)))
        texts.append(ocr_piece(piece))

    (out_dir / f"{path.stem}.md").write_text("\n\n".join(texts), encoding="utf-8")
    print("done:", path.name)

Images obtained: 5
done: 24158401-Rental-Agreement.png
done: 95980236-Rental-Agreement.png
done: 36199312-Rental-Agreement.png
done: 54770958-Rental-Agreement.png
done: 54945838-Rental-Agreement.png


In [14]:
!cd /kaggle/working && zip -r ocr.zip ocr

  adding: ocr/ (stored 0%)
  adding: ocr/36199312-Rental-Agreement.md (deflated 49%)
  adding: ocr/54770958-Rental-Agreement.md (deflated 55%)
  adding: ocr/95980236-Rental-Agreement.md (deflated 52%)
  adding: ocr/24158401-Rental-Agreement.md (deflated 55%)
  adding: ocr/54945838-Rental-Agreement.md (deflated 51%)
